In [ ]:
from pathlib import Path
import json as json
import numpy as np
import matplotlib.pyplot as plt
import os
import sys
path = os.path.abspath("/path/to/Documents/code/labelnoise-on-UQ")
os.environ["PYTHONPATH"] = path + os.pathsep + os.environ.get("PYTHONPATH", "")
sys.path.insert(0, path)
from src.Utils.vocabulary import (
    TRAIN, TEST, 
    HARD_TARGET, MULTI_TARGET, 
    AU, EU, TU, 
    index2multitarget_ESR, index2target_PTB
)
from src.Utils.synthetize_exp_evol import ExperimentEvolution, load_evol_results

%load_ext autoreload
%autoreload 2

In [ ]:
# General constants
dataset_idx2class = {"MNIST": {k: f"Nb {k}" for k in range(10)}, "ESR": index2multitarget_ESR, "PTB": index2target_PTB}
dataset_colors = {"MNIST": "green", "ESR": "orange", "PTB": "blue"}
nr_colors = {0.: '#93c47dff', 0.2: '#ffd966ff', 0.4: '#e69138ff', 0.6:'#cc0000ff'}
target_names = {"MNIST": HARD_TARGET, "ESR 1DCNNTransf": MULTI_TARGET, "ESR 2DCNN": MULTI_TARGET, "PTB": HARD_TARGET}
target_K = {"MNIST": 10, "ESR 1DCNNTransf": 5, "ESR 2DCNN": 5, "PTB": 2}

axis_fontsize = 14
title_fontsize = 16
legend_fontsize = 12

UQ_COLORS = {TU: 'skyblue', AU: 'lightgreen', EU: 'mistyrose'}
UQ_NAMES = [TU, AU, EU]
UQ_NAMES_SHORT = {TU: 'TU', AU: 'AU', EU: 'EU'}
MODES = [TRAIN, TEST]

loss_viz_names = {"CE_nodropout": "CE", 
                  "CE_dropout": "CE + Dropout", 
                  "GCE_q0-4_nodropout": "GCE (q=0.4)", "GCE_q0-7_nodropout": "GCE (q=0.7)", "GCE_q1_nodropout": "GCE (q=1.0)", 
                  "GCE_q0-4_dropout": "GCE (q=0.4) + Dropout", "GCE_q0-7_dropout": "GCE (q=0.7) + Dropout", "GCE_q1_dropout": "GCE (q=1.0) + Dropout", 
                  "RCE_A-16_nodropout": "RCE (A=16)",
                 "balancedCE_dropout": "balanced CE + Dropout", "balancedCE_nodropout": "balanced CE", 
                 "balancedGCE_q0-4_nodropout": "balanced GCE (q=0.4)", "balancedGCE_q0-7_nodropout": "balanced GCE (q=0.7)", "balancedGCE_q1_nodropout": "balanced GCE (q=1.0)",
                 "balancedGCE_q0-4_dropout": "balanced GCE (q=0.4) + Dropout", "balancedGCE_q0-7_dropout": "balanced GCE (q=0.7) + Dropout", "balancedGCE_q1_dropout": "balanced GCE (q=1.0) + Dropout",
                 "balancedRCE_A-16_nodropout": "balanced RCE (A=16)",
            }
dataset_viz_names ={
    "MNIST": "MNIST",
    "ESR 1DCNNTransf": "ESR 1D",
    "ESR 2DCNN": "ESR 2D",
    "PTB": "PTB"
}

def aggregate_prediction(pred):
    # pred: (n_classes, n_inference)
    mean_logits = np.mean(pred, axis=1)  # shape (n_classes,)
    return np.argmax(mean_logits)        # final predicted class


In [ ]:
# Gather paths

exp_root_path = Path("/path/to/experiments/publi_UQandNoise/ExpMem")
exp_summary = {
    "MNIST": ["CE_dropout", "CE_nodropout", "GCE_q0-4_nodropout", "GCE_q0-4_dropout", "GCE_q0-7_nodropout","GCE_q1_nodropout"],
    "ESR 1DCNNTransf": ["balancedCE_dropout", "balancedCE_nodropout", "balancedGCE_q0-4_nodropout", "balancedGCE_q0-4_dropout", "balancedGCE_q0-7_nodropout", "balancedGCE_q1_nodropout"],
    "ESR 2DCNN": ["balancedCE_dropout", "balancedCE_nodropout", "balancedGCE_q0-4_nodropout", "balancedGCE_q0-4_dropout", "balancedGCE_q0-7_nodropout", "balancedGCE_q1_nodropout"],
    "PTB": ["CE_dropout", "CE_nodropout", "GCE_q0-4_nodropout", "GCE_q0-4_dropout", "GCE_q0-7_nodropout","GCE_q1_nodropout"]
}
exp_roots = {
    "MNIST": ("MNIST", "Exp_ClassifMNIST_2DCNN_"),
    "ESR 1DCNNTransf": ("ESR_1DCNNTransf", "Exp_ClassifESRmulti_1DCNNTransf_"),
    "ESR 2DCNN": ("ESR_2DCNN", "Exp_ClassifESRmulti_2DCNN_"),
    "PTB": ("PTB", "Exp_ClassifPTBsub_1DCNNTransf_")
}
exp_paths = {
    d_name : {
        l_name : {
            nr : exp_root_path/f"{exp_roots[d_name][0]}"/f"{exp_roots[d_name][1]}{l_name}"/f"noise_rate_{str(nr).replace('.', '-')}" for nr in [0., 0.2, 0.4, 0.6]
            } for l_name in exp_summary[d_name]
        } for d_name in exp_summary.keys()
}

nb_runs_per_exp = 10
run_paths = {
    d: 
        {l: 
            {nr: [exp_paths[d][l][nr]/f"run-{i}" for i in range(nb_runs_per_exp) if (exp_paths[d][l][nr]/f"run-{i}").exists()]
             for nr in exp_paths[d][l].keys()
             } 
            for l in exp_paths[d].keys()} 
        for d in exp_paths.keys()
    }

dataset_names = ["MNIST", "ESR 1DCNNTransf", "ESR 2DCNN", "PTB"]
# dataset_names = ["ESR 1DCNNTransf", "ESR 2DCNN"]
n_dataset = len(dataset_names)
nr_rates = [0., 0.2, 0.4, 0.6]
modes = [TRAIN, TEST]
n_nr = len(nr_rates)
n_modes = len(modes)

# Predictions and metric without several inference with dropout
pred_name = "predictions.pkl"
metric_name = "metric.json"

# Predictions and metric without several inference with dropout
pred_name_MCDO = "predictions_MCdropout-entropy_ni5.pkl"
metric_name_MCDO = "metric_MCdropout-entropy_ni5.json"

# included_losses = {"MNIST": ["CE_nodropout", "CE_dropout"], 
#                    "ESR 1DCNNTransf": ["balancedCE_nodropout", "balancedCE_dropout"],
#                    "ESR 2DCNN": ["balancedCE_nodropout", "balancedCE_dropout"], 
#                    "PTB": ["CE_nodropout", "CE_dropout"]
#                    }

included_losses = {"MNIST": ["CE_nodropout", "GCE_q0-4_nodropout", "GCE_q0-7_nodropout", "GCE_q1_nodropout"], 
                   "ESR 1DCNNTransf": ["balancedCE_nodropout", "balancedGCE_q0-4_nodropout", "balancedGCE_q0-7_nodropout", "balancedGCE_q1_nodropout"],
                   "ESR 2DCNN": ["balancedCE_nodropout", "balancedGCE_q0-4_nodropout", "balancedGCE_q0-7_nodropout", "balancedGCE_q1_nodropout"], 
                   "PTB": ["CE_nodropout", "GCE_q0-4_nodropout", "GCE_q0-7_nodropout", "GCE_q1_nodropout"]
                   }

In [ ]:
# Print number of runs per dataset, loss and noise rate
for d_name in dataset_names:
    for l_name in exp_paths[d_name].keys():
        for nr in exp_paths[d_name][l_name].keys():
            print(f"{d_name} - {l_name} - nr {nr}: {len(run_paths[d_name][l_name][nr])} run(s)")
            print(run_paths[d_name][l_name][nr])

# Load evolution curves and configs

In [ ]:
# Create evolution summary if not created yet
for d_name in dataset_names:
    print(d_name)
    for l_name in included_losses[d_name]:
        print(l_name)
        print(exp_paths[d_name].keys())
        for nr in exp_paths[d_name][l_name].keys():
            for run_path in run_paths[d_name][l_name][nr]:
                if not (run_path/"evolution_summary").exists():
                    print(f"Missing evolution summary for {d_name} - {l_name} - nr {nr} - run {run_path.name}")
                    evolution = ExperimentEvolution(run_path, target_names[d_name])
                    evolution.get_score_evolution()
                    evolution.get_acc_evolution()
                    evolution.get_loss_evolution()
                    evolution.store_results()

In [ ]:
mode = TRAIN
scores, accuracies, losses, checkpoints = {}, {}, {}, {}
for d_name in dataset_names:
    scores[d_name] = {}
    accuracies[d_name] = {}
    losses[d_name] = {}
    checkpoints[d_name] = {}
    for l_name in included_losses[d_name]:
        scores[d_name][l_name] = {}
        accuracies[d_name][l_name] = {}
        losses[d_name][l_name] = {}
        for nr in exp_paths[d_name][l_name].keys():
            scores[d_name][l_name][nr] = []
            accuracies[d_name][l_name][nr] = []
            losses[d_name][l_name][nr] = []
            for run_path in run_paths[d_name][l_name][nr]:
                scores_run, accuracies_run, loss_run = {}, {}, {}
                print("run_path loaded:", run_path)
                scores_run[mode], accuracies_run[mode], loss_run[mode], checkpoint = load_evol_results(run_path, mode=mode)
                scores[d_name][l_name][nr].append(scores_run)
                accuracies[d_name][l_name][nr].append(accuracies_run)
                losses[d_name][l_name][nr].append(loss_run)
                print(f"Adding loss for {d_name}-{l_name}-{nr}: {loss_run[mode]}")
                print(f"Adding accuracies for {d_name}-{l_name}-{nr}: {accuracies_run[mode]}")
                if l_name not in checkpoints[d_name].keys():
                    print(f"Adding checkpoint for {d_name}, {len(checkpoint)}")
                    checkpoints[d_name][l_name] = checkpoint

scores
d_name:
    l_name:
        nr:
            [
                (run i) 
                'train': {
                    {'score_class_k':{
                        'noisy': list(array(N samples) x N_epochs)
                        'clean':list(array(N samples) x N_epochs)
                    },
                    ...
                    },
                    ...},
                'test: {...}
            ]

accuracies
d_name:
    l_name:
        nr:
            [
                (run 0)
                'train': {
                    {
                        'accuracy':{
                        'noisy': [float]
                        'clean': [float]
                        },
                        'accuracy_with_noise': [float]
                    }
                },
                'test: {}
            ]

# Visualize all trends during training

In [ ]:
for d_name in dataset_names:
    
    loss_names = included_losses[d_name]
    if len(loss_names) == 0:
        print(f"No loss to plot for {d_name}.")
        continue
    n_classes = 2 #target_K[d_name]
    fig, axs = plt.subplots(3, len(included_losses[d_name]),  figsize=(4 * len(included_losses[d_name]), 2 * 3), sharey='row')
    for i_l, l_name in enumerate(loss_names):
        print(checkpoints[d_name].keys())
        checkpoint_epochs = checkpoints[d_name][l_name]
        print("Checkpoint epochs shape:", len(checkpoint_epochs))
        for i_nr, nr in enumerate(nr_rates):
            print(f"Plotting {d_name} - {l_name} - nr {nr}")
            mode = TRAIN            
            #----------- First line: accuracy evolution during training, one for clean, one for noisy and one for noisy with noise called "memorized"
            for s_type in ["noisy", "clean"]:
                by_run_acc = [np.array(accuracies[d_name][l_name][nr][i_run][mode]['accuracy'][s_type]) for i_run in range(len(run_paths[d_name][l_name][nr]))]
                if all(by_run.size == 0 for by_run in by_run_acc):
                    print(f"Skipping {d_name} - {l_name} - nr {nr} - {s_type} because all runs have empty accuracy arrays.")
                    continue
                acc_mean = np.mean(np.stack(by_run_acc), axis=0)
                acc_std = np.std(np.stack(by_run_acc), axis=0)
                print("Accuracy shapes:", acc_mean.shape, acc_std.shape)
                if len(checkpoint_epochs) == len(acc_mean):
                    axs[0, i_l].plot(checkpoint_epochs, acc_mean, '--' if s_type == 'noisy' else '-', color=nr_colors[nr], label=f"nr {nr}, {s_type}")
                    axs[0, i_l].fill_between(checkpoint_epochs, acc_mean - acc_std, acc_mean + acc_std, color=nr_colors[nr], alpha=0.2)
                    # axs[0, i_l].plot(acc_mean, '--' if s_type == 'noisy' else '-', color=nr_colors[nr], label=f"nr {nr}, {s_type}")
                    # axs[0, i_l].fill_between(acc_mean - acc_std, acc_mean + acc_std, color=nr_colors[nr], alpha=0.2)
            
            # Add a line for the accuracy with noise
            by_run_acc2noise = [np.array(accuracies[d_name][l_name][nr][i_run][mode]['accuracy_with_noise']["noisy"]) for i_run in range(len(run_paths[d_name][l_name][nr]))]
            if not all(by_run.size == 0 for by_run in by_run_acc2noise):
                acc2noise_mean = np.mean(np.stack(by_run_acc2noise), axis=0)
                acc2noise_std = np.std(np.stack(by_run_acc2noise), axis=0)
                print("Accuracy with noise shapes:", acc2noise_mean.shape, acc2noise_std.shape)
                axs[0, i_l].plot(checkpoint_epochs, acc2noise_mean, '-.', color=nr_colors[nr], label=f"nr {nr}, Memorization")
                # axs[0, i_l].plot(acc2noise_mean, '-.', color=nr_colors[nr], label=f"nr {nr}, Memorization")
            
            # axs[0, i_l].set_xlabel("Epoch", fontsize=axis_fontsize)
            axs[0, i_l].set_ylabel("Accuracy", fontsize=axis_fontsize)
            axs[0, i_l].set_title(f"{dataset_viz_names[d_name]} {mode}\n{loss_viz_names[l_name]}", fontsize=title_fontsize)
            
            #--------- Second line: max score evolution during training, one color by noise rate, mean across runs, with std shading
            for s_type in ["noisy", "clean"]:
                by_run_matrices = [np.stack(run_score[mode][f'score_class_{n_classes - 1}'][s_type]) for run_score in scores[d_name][l_name][nr]] # run_score[...] is a list of (N_samples,) arrays. stacked shape: (N_epochs, N_samples)
                by_run_mean = [np.mean(elem, axis=1) for elem in by_run_matrices] # list of arrays, each with shape: (N_epochs,)
                by_run_mean = np.stack(by_run_mean) # shape: (N_runs, N_epochs)
                # Mean and std across runs
                mean_scores = np.mean(by_run_mean, axis=0)
                std_scores = np.std(by_run_mean, axis=0)
                print("Max score shapes:", mean_scores.shape, std_scores.shape)
                axs[1, i_l].plot(checkpoint_epochs, mean_scores, '--' if s_type == 'noisy' else '-', color=nr_colors[nr], label=f"nr {nr}, {s_type}")
                # axs[1, i_l].plot(mean_scores, '--' if s_type == 'noisy' else '-', color=nr_colors[nr], label=f"nr {nr}, {s_type}")
                axs[1, i_l].fill_between(checkpoint_epochs, mean_scores - std_scores, mean_scores + std_scores, color=nr_colors[nr], alpha=0.2)

            axs[1, i_l].set_xlabel("Epoch", fontsize=axis_fontsize)
            axs[1, i_l].set_ylabel("Max score", fontsize=axis_fontsize)
            

            #----------- Third line: Loss evolution
            by_run_loss = [np.array(losses[d_name][l_name][nr][i_run][mode]['loss']) for i_run in range(len(run_paths[d_name][l_name][nr]))]
            # print("by run loss", len(by_run_loss), [elem.shape for elem in by_run_loss])
            loss_mean = np.mean(np.stack(by_run_loss), axis=0)
            loss_std = np.std(np.stack(by_run_loss), axis=0)
            print("Loss shapes:", loss_mean.shape, loss_std.shape)
            axs[2, i_l].plot(loss_mean, color=nr_colors[nr], label=f"nr {nr}")
            axs[2, i_l].fill_between(range(len(loss_mean)), loss_mean - loss_std, loss_mean + loss_std, color=nr_colors[nr], alpha=0.2)
            axs[2, i_l].set_xlabel("Epoch", fontsize=axis_fontsize)
            axs[2, i_l].set_ylabel("Loss", fontsize=axis_fontsize)
                
                
        axs[0, len(included_losses[d_name]) - 1].legend(fontsize=legend_fontsize, bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
    plt.show()

## Publication figures

In [ ]:
from matplotlib.lines import Line2D
legend_fontsize = 12
title_fontsize = 18
axis_fontsize = 14
for d_name in dataset_names:
    
    loss_names = included_losses[d_name]
    if len(loss_names) == 0:
        print(f"No loss to plot for {d_name}.")
        continue
    n_classes = 2 #target_K[d_name]
    fig, axs = plt.subplots(2, len(included_losses[d_name]),  figsize=(3.5 * len(included_losses[d_name]), 2 * 2.5), sharey='row')
    for i_l, l_name in enumerate(loss_names):
        checkpoint_epochs = checkpoints[d_name][l_name]
        for i_nr, nr in enumerate(nr_rates):
            if nr == 0.6 and d_name == "PTB":
                print(f"Skipping {d_name} - {l_name} - nr {nr}")
                continue
            print(f"Plotting {d_name} - {l_name} - nr {nr}")
            mode = TRAIN            
            #----------- First line: accuracy evolution during training, one for clean, one for noisy and one for noisy with noise called "memorized"
            for s_type in ["noisy", "clean"]:
                by_run_acc = [np.array(accuracies[d_name][l_name][nr][i_run][mode]['accuracy'][s_type]) for i_run in range(len(run_paths[d_name][l_name][nr]))]
                # print("by run acc", s_type, len(by_run_acc), [elem.shape for elem in by_run_acc])
                if all(by_run.size == 0 for by_run in by_run_acc):
                    print(f"Skipping {d_name} - {l_name} - nr {nr} - {s_type} because all runs have empty accuracy arrays.")
                    continue
                acc_mean = np.mean(np.stack(by_run_acc), axis=0)
                acc_std = np.std(np.stack(by_run_acc), axis=0)
                axs[0, i_l].plot(checkpoint_epochs, acc_mean, '--' if s_type == 'noisy' else '-', color=nr_colors[nr], label=f"nr {nr}, {s_type}")
                axs[0, i_l].fill_between(checkpoint_epochs, acc_mean - acc_std, acc_mean + acc_std, color=nr_colors[nr], alpha=0.1)
            
            # Add a line for the accuracy with noise
            by_run_acc2noise = [np.array(accuracies[d_name][l_name][nr][i_run][mode]['accuracy_with_noise']["noisy"]) for i_run in range(len(run_paths[d_name][l_name][nr]))]
            if not all(by_run.size == 0 for by_run in by_run_acc2noise):
                acc2noise_mean = np.mean(np.stack(by_run_acc2noise), axis=0)
                acc2noise_std = np.std(np.stack(by_run_acc2noise), axis=0)
                axs[0, i_l].plot(checkpoint_epochs, acc2noise_mean, '-.', color=nr_colors[nr], label=f"nr {nr}, Memorization")
                axs[0, i_l].fill_between(checkpoint_epochs, acc2noise_mean - acc2noise_std, acc2noise_mean + acc2noise_std, color=nr_colors[nr], alpha=0.1)

            # axs[0, i_l].set_xlabel("Epoch", fontsize=axis_fontsize)
            axs[0, i_l].set_ylabel("Accuracy", fontsize=axis_fontsize)
            axs[0, i_l].set_title(f"{loss_viz_names[l_name]}", fontsize=title_fontsize)
            
            #--------- Second line: max score evolution during training, one color by noise rate, mean across runs, with std shading
            for s_type in ["noisy", "clean"]:
                by_run_matrices = [np.stack(run_score[mode][f'score_class_{n_classes - 1}'][s_type]) for run_score in scores[d_name][l_name][nr]] # run_score[...] is a list of (N_samples,) arrays. stacked shape: (N_epochs, N_samples)
                print("by run matrices", s_type, len(by_run_matrices), [elem.shape for elem in by_run_matrices])
                by_run_mean = [np.mean(elem, axis=1) for elem in by_run_matrices] # list of arrays, each with shape: (N_epochs,)
                by_run_mean = np.stack(by_run_mean) # shape: (N_runs, N_epochs)
                # Mean and std across runs
                mean_scores = np.mean(by_run_mean, axis=0)
                std_scores = np.std(by_run_mean, axis=0)
                axs[1, i_l].plot(checkpoint_epochs, mean_scores, '--' if s_type == 'noisy' else '-', color=nr_colors[nr], label=f"nr {nr}, {s_type}")
                axs[1, i_l].fill_between(checkpoint_epochs, mean_scores - std_scores, mean_scores + std_scores, color=nr_colors[nr], alpha=0.1)

            axs[1, i_l].set_xlabel("Epoch", fontsize=axis_fontsize)
            axs[1, i_l].set_ylabel("Max score", fontsize=axis_fontsize)
        
    # Legend
    handles = [
        Line2D([0], [0], color=nr_colors[nr], lw=2, label=r"$\eta=nr$".replace("nr", f"{nr}")) for nr in nr_rates
    ]
    handles.append(Line2D([0], [0], color='black', lw=2, ls='-', label="Clean samples"))
    handles.append(Line2D([0], [0], color='black', lw=2, ls='--', label="Noisy samples"))
    handles.append(Line2D([0], [0], color='black', lw=2, ls='-.', label="Memorization"))
    labels = [r"$\eta=nr$".replace("nr", f"{nr}") for nr in nr_rates] + ["Clean samples", "Noisy samples", "Memorization"]
    legend_order = [0, 4, 1, 5, 2, 6, 3]  # Order of legend entries
    handles = [handles[i] for i in legend_order]
    labels = [labels[i] for i in legend_order]
    fig.legend(
        handles,
        labels,
        loc="lower center",
        ncol=4,
        fontsize=legend_fontsize,
        bbox_to_anchor=(0.5, -0.02)
    )

    # Let Matplotlib compute all text/artist positions before saving
    fig.canvas.draw()

    # Adjust subplot positions while leaving room for the legend
    fig.tight_layout(rect=[0, 0.12, 1, 1])

    # Save the complete figure, including the external legend
    fig.savefig(
        exp_root_path / exp_roots[d_name][0] /
        f"mem_evolution_{d_name.replace(' ', '')}.svg",
        format="svg",
        bbox_inches="tight",
    )

    plt.show()
    plt.close(fig)
